# Weights configuration for Jaccard Index generation

In this notebook, I will be doing experimentation, training of models, and evaluation of different weight configurations for generating the Jaccard Index.


In [38]:
import psycopg
import optuna
import igraph as ig
from sshtunnel import SSHTunnelForwarder
import dotenv
import os
dotenv.load_dotenv()

True

In [39]:
server = SSHTunnelForwarder(
    os.getenv("SSH_HOST"),
    ssh_username=os.getenv("SSH_USERNAME"),
    ssh_pkey=os.getenv("SSH_PKEY_PATH"),
    remote_bind_address=('127.0.0.1', 5432)
)
server.start()
conn = psycopg.connect(f"dbname={os.getenv('DB_NAME')} user={os.getenv('DB_USER')} host=127.0.0.1 port={server.local_bind_port} password={os.getenv('DB_PASSWORD')}")
conn.read_only = True
cur = conn.cursor()


In [40]:
START_DATE = '2026-08-27'
END_DATE = '2026-09-01'
FEATURE_NAMES = [
    'method', 'host', 'query_part', 'path_part', 'body_part', 'header', 'user_agent',
    'request_length', 'status_code', 'server_protocol', 'server_name', 'remote_addr',
    'accept_encoding', 'content_type', 'scheme'
]

In [41]:
cur.execute('SELECT * FROM part_types;')
print([desc[0] for desc in cur.description])
part_types = cur.fetchall()
print(part_types)

['id', 'name', 'description']
[(UUID('07901ba7-64d3-4603-9f5c-38a761ea9ed6'), 'method', 'HTTP Method (GET, POST, etc.)'), (UUID('e1b77c40-928f-4a87-9130-5ce6aeb91b62'), 'host', 'The host of the request'), (UUID('d3b24ec8-85e9-414a-9639-0d4b489388d9'), 'query_part', 'Part of the query part of the URL'), (UUID('6ff317f1-d8ec-48f0-8c6a-5e146c799185'), 'path_part', 'Part of the endpoint path of the URL'), (UUID('a515a5f3-4354-4186-b264-f3b616ab8197'), 'body_part', 'Part of the body of the request'), (UUID('eaec0218-0f1d-487b-9b63-4543a6422e36'), 'header', 'HTTP Header (User-Agent, Content-Type, etc.)'), (UUID('2165835b-b1b1-4da3-bbf6-683d0e82856d'), 'user_agent', 'The user agent of the request'), (UUID('756a9262-c513-4cc4-9f3f-3730f4055d35'), 'request_length', 'Length of the HTTP request'), (UUID('d8894b05-3053-4c45-9f58-03ebb3ee2f15'), 'status_code', 'HTTP Status Code'), (UUID('8ac61f8d-81b3-4e29-bb9e-13bfb22974ed'), 'server_protocol', 'The protocol used by the server (HTTP/1.1, HTTP/2, e

In [42]:
query = """
WITH selected_requests AS (
    SELECT id
    FROM requests
    WHERE created_at BETWEEN %s AND %s
),
weights AS (
    SELECT x.name, x.weight
    FROM jsonb_to_recordset(%s::jsonb) AS x(name text, weight double precision)
), shared_parts AS (
    SELECT rp1.request_id AS request_1,
        rp2.request_id AS request_2,
        SUM(w.weight) AS shared_parts_count
    FROM request_to_parts rp1
    JOIN request_to_parts rp2 ON rp1.part_id = rp2.part_id
    JOIN request_parts p ON rp1.part_id = p.id
    JOIN part_types pt ON pt.id = p.part_type
    JOIN weights w ON w.name = pt.name
    JOIN selected_requests sr1 ON rp1.request_id = sr1.id
    JOIN selected_requests sr2 ON rp2.request_id = sr2.id
    WHERE p.excluded = FALSE
      AND rp1.request_id < rp2.request_id
    GROUP BY rp1.request_id, rp2.request_id
),
non_excluded_parts AS (
    SELECT r.id AS request_id, SUM(w.weight) AS non_excluded_count
    FROM selected_requests r
    JOIN request_to_parts rtp ON r.id = rtp.request_id
    JOIN request_parts rp ON rtp.part_id = rp.id
    JOIN part_types pt ON pt.id = rp.part_type
    JOIN weights w ON w.name = pt.name
    WHERE rp.excluded = FALSE 
    GROUP BY r.id
),
full_graph AS (
    SELECT r1.ext_id AS request_1,
        r2.ext_id AS request_2,
        sp.shared_parts_count::float / (
            nep1.non_excluded_count + nep2.non_excluded_count - sp.shared_parts_count
        )::float AS jaccard_index
    FROM shared_parts sp
    JOIN requests r1 ON sp.request_1 = r1.id
    JOIN requests r2 ON sp.request_2 = r2.id
    JOIN non_excluded_parts nep1 ON sp.request_1 = nep1.request_id
    JOIN non_excluded_parts nep2 ON sp.request_2 = nep2.request_id
)
SELECT * FROM full_graph
WHERE jaccard_index > (SELECT value::float FROM loggraph_config WHERE key = 'min_jaccard_index');
"""
data = cur.execute(query,
                   (START_DATE, END_DATE,
                    psycopg.types.json.Jsonb([{"name": name, "weight": 1.0} for name in FEATURE_NAMES])
                    )).fetchall()
print(len(data))

1773820


Now we will build the graph: for each request, we will create nodes for its parts and connect them based on their relationships. Expected row count 1773820.


In [ ]:
TARGET_CLUSTERS = 50
MIN_WEIGHT = 0.0
MAX_WEIGHT = 5.0
WEIGHT_STEP = 0.1

def target_function(num_clusters: int):
    return (num_clusters - TARGET_CLUSTERS) ** 2

def evaluate_weights(weights: list[dict[str, float]]) -> tuple[float, int]:
    data = cur.execute(query, (
        START_DATE, END_DATE,
        psycopg.types.json.Jsonb(
            weights
        )
    )).fetchall()
    graph = ig.Graph.TupleList(data, directed=False, edge_attrs=['weight'])
    clusters = graph.community_leiden(weights='weight')
    cluster_count = len(clusters)
    score = target_function(cluster_count)
    return score, cluster_count

def objective(trial: optuna.Trial) -> float:
    trial_weights = [
        {
            "name": name,
            "weight": trial.suggest_float(name, MIN_WEIGHT, MAX_WEIGHT, step=WEIGHT_STEP)
        }
        for name in FEATURE_NAMES
    ]
    score, cluster_count = evaluate_weights(trial_weights)
    trial.set_user_attr('cluster_count', cluster_count)
    return score

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_weights = [{"name": name, "weight": float(value)} for name, value in study.best_params.items()]
best_score, best_cluster_count = evaluate_weights(best_weights)

print(f'Best score: {best_score}')
print(f'Best cluster count: {best_cluster_count}')
print('Best weights:')
print(best_weights)


[I 2026-11-22 23:57:05,984] A new study created in memory with name: no-name-1271cb17-1d29-422c-9e12-27cb121dd0e4
  0%|          | 0/50 [00:00<?, ?it/s]